# 02. Kiểm tra hash đồ thị của src/build_graph.py (task 0.4, tiêu chí 1)

Dựng lại đồ thị NF-ToN-IoT-v2 (W = S = 1000 flow, chia `chronological` 70/15/15) bằng module và so với hash của notebook Tuần 1 (`cb8842d60f9a3a5a`, 16.941 đồ thị). Dữ liệu: `athena21/netflow-datasets-v2`, file `NF-ToN-IoT-v2.csv` 2.676.491.000 byte, trùng dung lượng file Phong dùng (2552,5 MB).

In [ ]:
import glob, os, subprocess, sys, time

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-3000:], end="")
    print(r.stderr[-3000:], end="", file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/kaggle/working/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])
sys.path.insert(0, REPO)

In [ ]:
import pandas as pd, numpy as np, torch, torch_geometric
from src import build_graph as bg
print("pandas", pd.__version__, "| numpy", np.__version__, "| torch", torch.__version__, "| PyG", torch_geometric.__version__)
CSV = next(f for f in glob.glob("/kaggle/input/**/NF-ToN-IoT-v2.csv", recursive=True))
print(CSV, os.path.getsize(CSV), "byte")

t0 = time.time()
flows, classes, log = bg.load_clean("nf_ton_iot_v2", CSV)
print(f"Làm sạch {time.time() - t0:.0f} s | lớp: {classes}")
print(log.to_string())

In [ ]:
t0 = time.time()
graphs, split, scaler, _ = bg.build_dataset(flows, 1000, 1000)
fp = bg.fingerprint(graphs)
EXPECTED = "cb8842d60f9a3a5a"
print(f"Dựng {len(graphs):,} đồ thị trong {time.time() - t0:.0f} s")
print({p: len(w) for p, w in split.items()})
print("Hash:", fp[:16], "| notebook Tuần 1:", EXPECTED, "| trùng:", fp.startswith(EXPECTED))
print("Hash đầy đủ:", fp)
assert fp.startswith(EXPECTED), "hash khác notebook Tuần 1"